# 基于静态 Tensor 的 Vector 算子入门

本节以 `z = x + y` 为例，介绍由开发者显式管理 UB 空间和流水同步的 **Static Tensor（静态 Tensor）编程方式**。

## 学习目标

- 用 `LocalMemAllocator<Hardware::UB>` 线性分配器规划本地内存。
- 用 `SetFlag/WaitFlag` 处理跨流水依赖，用 `PipeBarrier` 处理流水屏障。
- 了解芯片架构，理解 GM、UB、L1、L0A、L0B、L0C 等 buffer 在硬件上的位置和关系。

## 1. 静态 Tensor 的基本特点

静态 Tensor 编程遵循基础算子的三段式流程：

```cpp
CopyIn();   // 数据搬运
Compute();  // 本地计算
CopyOut();  // 数据搬出
```

与更高层封装不同，静态 Tensor 代码需要直接描述每个阶段使用哪块本地存储、由哪条硬件流水写入、由哪条硬件流水读取，以及什么时候可以复用 buffer。开发者需要把 Tensor 的物理位置、数据依赖和同步关系显式写在代码中。

静态 Tensor 的基本计算流程如下（以 Vector 算子为例）：

![静态 Tensor 基本计算流程](images/02.05/1_basic_compute_flow.svg)

主要流程如下：

1. 使用 `DataCopy`（MTE2 流水）将输入数据从 GM 搬运到 UB 的输入 Tensor；
2. 在 Vector 读取输入 Tensor 前，等待 MTE2 搬入流水完成；
3. Vector 单元读取 UB 输入 Tensor，完成计算并将结果写回 UB 输出 Tensor；
4. 在 MTE3 搬出输出 Tensor 前，等待 Vector 计算流水完成；
5. 使用 `DataCopy`（MTE3 流水）将 UB 输出 Tensor 的结果写回 GM。

主要特征是：**Tensor 物理位置、同步关系和 buffer 复用时机需要在代码中显式指定**。因此静态 Tensor 更适合需要精细控制本地存储和硬件流水的计算场景。

静态 Tensor kernel 还需要在`__global__`函数入口和退出位置完成基础运行状态处理：

```cpp
AscendC::InitSocState();
// kernel主体：GM绑定、本地Tensor分配、搬运、计算、写回
AscendC::PipeBarrier<PIPE_ALL>();
```

`InitSocState()` 用于初始化全局状态寄存器；kernel 结束前的 `PipeBarrier<PIPE_ALL>()` 用于等待已发出的流水指令完成，避免 kernel 退出时仍有未完成的本地计算或搬运操作。


## 2. 内存分配

静态 Tensor 编程中，通过 `LocalMemAllocator` 在指定硬件位置线性分配本地内存，返回用于存储和计算的 `LocalTensor`。

### 2.1 线性分配器 LocalMemAllocator

这种分配方式需要根据 Tensor 存储的物理位置，去创建对应的`LocalMemAllocator`对象。

再通过`Alloc`方法，创建对应物理位置的LocalTensor去存储和计算数据。

```cpp
// tileLen非静态，LocalMemAllocator分配Buffer空间场景
AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator; // 创建UB位置的LocalMemAllocator
AscendC::LocalTensor<float> xLocal = ubAllocator.Alloc<float>(tileLen); // 创建元素个数为tileLen的LocalTensor
AscendC::LocalTensor<float> yLocal = ubAllocator.Alloc<float>(tileLen); // 创建元素个数为tileLen的LocalTensor
AscendC::LocalTensor<float> zLocal = ubAllocator.Alloc<float>(tileLen); // 创建元素个数为tileLen的LocalTensor
```

分配器按调用顺序线性规划空间，不提供内存释放或自动流水同步。同一 kernel 生命周期内，同一种物理存储只使用一个分配器；应在循环外分配，在循环内通过同步安全复用。

当 tile 长度在编译期已知时，可使用 `ubAllocator.Alloc<float, TILE_LENGTH>()`，后面的可执行示例采用此形式。

以下是常用的`LocalMemAllocator`对象及其对应的位置。本节 Vector 示例只使用 UB，其他位置供理解存储层级：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">对象</th>
      <th align="left">常用场景</th>
      <th align="left">物理位置</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::UB&gt;</code></td>
      <td>在 UB 上分配 Vector 计算使用的本地 buffer</td>
      <td>Unified Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::L1&gt;</code></td>
      <td>在 L1 上分配 Cube 前级搬入 buffer</td>
      <td>L1 Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::L0A&gt;</code></td>
      <td>在 L0A 上分配矩阵 A 分形数据</td>
      <td>L0A Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::L0B&gt;</code></td>
      <td>在 L0B 上分配矩阵 B 分形数据</td>
      <td>L0B Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::BIAS&gt;</code></td>
      <td>在 BiasTable Buffer 上分配 Bias 数据</td>
      <td>BT Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::L0C&gt;</code></td>
      <td>在 L0C 上分配 Cube 累加结果</td>
      <td>L0C Buffer</td>
    </tr>
  </tbody>
</table>


### 2.2 本课示例的 UB 空间用量

本例每个 tile 包含 256 个 float，使用同一个 UB 分配器依次分配三个 Tensor：

```cpp
constexpr uint32_t TILE_LENGTH = 256;
AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
auto xLocal = ubAllocator.Alloc<float, TILE_LENGTH>();
auto yLocal = ubAllocator.Alloc<float, TILE_LENGTH>();
auto zLocal = ubAllocator.Alloc<float, TILE_LENGTH>();
```

`Alloc` 的长度表示元素个数。每个 Tensor 占 256 × 4 = 1024 字节，三个 Tensor 合计占 3072 字节。分配器依次为它们规划互不重叠的空间，开发者需要确保总用量不超过可用 UB 容量。

`GlobalTensor` 不负责申请 GM 内存，它在 kernel 中绑定 Host 侧传入的 GM 地址。


## 3. 流水同步

AI Core 内部存在多条可以并行发射的硬件指令流水队列。以基础 Vector 场景为例，一轮计算通常涉及三类流水：

- `MTE2`：从 GM 搬入数据到 UB；
- `V`：读取 UB 中的数据执行 Vector 计算，并将结果写回 UB；
- `MTE3`：将 UB 输出结果搬回 GM。

这些流水可以异步并行运行。并行能够提升吞吐，但当不同流水读写同一片 Local Memory 时，单纯依赖代码先后顺序并不能保证硬件执行时序。例如，代码中先写了 `DataCopy(GM → UB)`，再写 `Add`，并不意味着 Vector 读取 UB 时 MTE2 一定已经完成搬入；同理，代码中先写 `Add`，再写 `DataCopy(UB → GM)`，也不意味着 MTE3 搬出时 Vector 一定已经完成写入。

![MTE2 / V / MTE3 流水同步关系](images/02.05/3_pipeline_sync.svg)

因此，静态 Tensor 中的同步需要围绕真实数据依赖来判断：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">依赖类型</th>
      <th align="left">典型场景</th>
      <th align="left">同步目标</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>跨流水写后读</td>
      <td>MTE2 写 UB 后，Vector 读 UB</td>
      <td>读之前等待数据搬运完成</td>
    </tr>
    <tr>
      <td>跨流水写后读/读后写</td>
      <td>Vector 写 UB 后，MTE3 读 UB；MTE3 读完后下一轮复用 UB</td>
      <td>复用或搬出前等待前序流水完成</td>
    </tr>
    <tr>
      <td>同一流水内部依赖</td>
      <td>后一条 Vector 指令依赖前一条 Vector 指令结果</td>
      <td>后序指令开始前等待前序指令读写完成</td>
    </tr>
  </tbody>
</table>

开发者可以使用两类接口插入同步：

1. `PipeBarrier<PIPE_*>`：约束同一类流水内部的指令顺序；
2. `SetFlag/WaitFlag`：约束不同硬件流水之间的指令顺序。


### 3.1 `PipeBarrier`：同一流水内部的顺序依赖

`PipeBarrier` 用于约束同一类流水内的指令顺序。同一流水中的指令虽然按顺序发射，但后一条指令开始执行时，前一条指令的数据读写不一定已经全部完成；如果后一条指令依赖前一条指令写出的结果，就需要插入对应流水的 `PipeBarrier`。

以 Vector 计算为例：

```cpp
AscendC::Mul(tmpLocal, xLocal, xLocal, tileLen);    // tmp = x * x
AscendC::PipeBarrier<PIPE_V>();
AscendC::Add(zLocal, tmpLocal, yLocal, tileLen);    // z = tmp + y
```

这里的屏障保护的是 `tmpLocal` 上的中间结果。如果没有 `PipeBarrier<PIPE_V>()`，后一条 Vector 指令可能在前一条指令结果尚未完全可见时读取 `tmpLocal`。

`PipeBarrier` 的使用原则是：当同一流水中的后一条指令依赖前一条指令写出的结果时，需要显式保证顺序；如果两条指令之间没有数据依赖，过度插入屏障会降低流水并行度。


### 3.2 `SetFlag/WaitFlag`：跨流水同步

`SetFlag/WaitFlag` 用于表达不同硬件流水之间的同步。它本质上是在源流水和目的流水之间建立一组“锁”：

- `SetFlag`：源流水执行到该指令时，会等待源流水前序指令的读写操作完成，然后设置对应事件标志；
- `WaitFlag`：目的流水执行到该指令时，如果事件标志尚未设置，会阻塞目的流水后续指令；事件满足后清除标志并继续执行。

基础 Add 样例中的跨流水同步如下：

```cpp
constexpr event_t eventMte2ToV = EVENT_ID0;
constexpr event_t eventVToMte3 = EVENT_ID1;

AscendC::DataCopy(xLocal, xGm[offset], tileLen);
AscendC::DataCopy(yLocal, yGm[offset], tileLen);
AscendC::SetFlag<AscendC::HardEvent::MTE2_V>(eventMte2ToV);
AscendC::WaitFlag<AscendC::HardEvent::MTE2_V>(eventMte2ToV);

AscendC::Add(zLocal, xLocal, yLocal, tileLen);
AscendC::SetFlag<AscendC::HardEvent::V_MTE3>(eventVToMte3);
AscendC::WaitFlag<AscendC::HardEvent::V_MTE3>(eventVToMte3);

AscendC::DataCopy(zGm[offset], zLocal, tileLen);
```

这两个事件分别保护两段依赖：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">同步点</th>
      <th align="left">数据流</th>
      <th align="left">保护的数据</th>
      <th align="left">缺失风险</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>MTE2_V</code></td>
      <td><code>DataCopy</code> 搬入 → Vector 计算</td>
      <td>UB 输入 <code>xLocal</code>、<code>yLocal</code></td>
      <td>Vector 可能读到尚未搬完的数据</td>
    </tr>
    <tr>
      <td><code>V_MTE3</code></td>
      <td>Vector 计算 → <code>DataCopy</code> 写回</td>
      <td>UB 输出 <code>zLocal</code></td>
      <td>MTE3 可能写回尚未计算完成的数据</td>
    </tr>
  </tbody>
</table>

如果下一轮循环要复用 `xLocal`、`yLocal` 或 `zLocal`，还需要确认上一轮搬入、计算和搬出对这些 buffer 的访问已经结束。例如复用输出 `zLocal` 前，需要保证上一轮 `MTE3` 已经完成对 `zLocal` 的读取。


### 3.3 反向同步：保护下一轮要覆盖的 buffer

只有正向同步仍不够。下一轮 MTE2 可能过早覆盖输入，下一轮 Vector 也可能过早覆盖尚未搬出的输出。

| 事件方向 | 等待位置 | 防止的问题 |
| --- | --- | --- |
| MTE2_V | 本轮 Add 前 | 输入还没搬完就参与计算 |
| V_MTE3 | 本轮写回前 | 结果还没算完就搬出 |
| V_MTE2 | 下一轮搬入前 | 覆盖上轮 Vector 尚未读完的输入 |
| MTE3_V | 下一轮计算前 | 覆盖上轮 MTE3 尚未读完的输出 |

完整代码会在首轮跳过反向等待，因为没有上轮任务；末轮不再发出给下一轮的反向事件，避免残留未消费标志。首末轮重合的单 tile 情况也要正确处理。


## 4. 数据搬入

数据搬入阶段负责将 GM 输入搬到本地存储。基础 Vector 静态 Tensor 通常将输入搬入 UB：

```cpp
AscendC::DataCopy(xLocal, xGm[tileOffset], tileLen);
```

`DataCopy` 最常见的形态是连续搬运：源地址和目的地址各自连续存放，`count` 参数表示搬运的元素个数，接口会根据数据类型换算成字节数。基础 Add 样例中的输入向量就是连续向量，适合使用这种最简单的搬运方式。

数据搬运时需要关注地址和长度是否满足硬件对齐要求。常见规则可以按以下方式理解：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">场景</th>
      <th align="left">推荐接口</th>
      <th align="left">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>连续且满足对齐要求</td>
      <td><code>DataCopy(dstLocal, srcGm, count)</code></td>
      <td>连续对齐搬运，适合本节 Add 样例</td>
    </tr>
    <tr>
      <td>多段连续块，块间存在固定间隔</td>
      <td>带 <code>DataCopyParams</code> 的 <code>DataCopy</code></td>
      <td>用 blockLen、blockCount、stride 描述分块搬运</td>
    </tr>
    <tr>
      <td>尾块或地址不满足对齐要求</td>
      <td><code>DataCopyPad</code> 或非对齐搬运接口</td>
      <td>按对应接口的地址与长度约束处理非 32B 整数倍及尾块搬运</td>
    </tr>
    <tr>
      <td>UB 内部重排或中间数据搬运</td>
      <td><code>Copy</code></td>
      <td>在 UB 内不同区域之间搬运，支持 mask 和间隔控制</td>
    </tr>
  </tbody>
</table>

搬运指令发出后，需要根据后续指令类型插入同步。若后续由 Vector 计算读取 `xLocal`，需要使用 `MTE2_V` 方向的事件同步：

![数据搬入与 MTE2_V 同步](images/02.05/4_pipeline_flow.svg)


## 5. 数据切分与搬运约束

课堂主案例为 ND 布局、float 类型、shape `(8, 2048)`，共 16384 个元素。启动 8 个 Vector 核，每核 2048 个元素，循环 8 次，每次处理 256 个元素。

```text
16384 元素 / 8 核 = 2048 元素/核
2048 元素 / 256 元素/tile = 8 tile/核
本核 GM 起点 = GetBlockIdx() × blockLength
本轮 GM 偏移 = i × TILE_LENGTH
```

这里直接把 `totalLength` 按值传入 kernel，Host 确保能够均匀分核和分 tile。不是任意长度的泛化实现。

本例 `DataCopy(..., count)` 的 count 是元素个数；连续搬运要求关注 32 字节对齐。256 个 float 为 1024 字节，本例 GM 偏移、UB 地址和搬运长度均满足要求。任意尾块不能仅把长度改小就假定搬运正确，需要按接口约束选择 `DataCopyPad` 等方式并处理有效元素。本节不展开非对齐尾块。

本例是单 buffer 教学实现。Double Buffer 需增加 ping/pong 两套地址、轮转选择和事件管理；静态 Tensor 不会自动启用双缓冲。

## 6. 环境准备与可执行工程

开始编写示例前，先执行下面的代码单元初始化 Jupyter 环境。代码会自动加载 CANN 环境变量并导入当前 Notebook 进程，同时创建 `Sources/02.05` 代码目录，供后续使用 bisheng 编译器编译和运行算子。仓库中的 `src/02.05` 保存相同源码。

In [ ]:
from pathlib import Path
import os
import subprocess

# 加载 CANN 环境，并将变量导入当前 Jupyter 进程。
set_env = Path(os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann")) / "set_env.sh"
if not set_env.is_file():
    set_env = Path("/usr/local/Ascend/cann/set_env.sh")

result = subprocess.run(
    ["bash", "-c", 'source "$1" && env -0', "bash", str(set_env)],
    capture_output=True,
    text=True,
    check=True,
)
for entry in result.stdout.split("\0"):
    if "=" in entry:
        key, value = entry.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")

Path("Sources/02.05").mkdir(parents=True, exist_ok=True)


### 6.1 Device 代码分步实现

按顺序执行下面的单元，将初始化、线性分配、搬入、计算和搬出代码写入同一个 `add_static.asc`。首个代码单元创建文件，后续单元追加代码；重新生成时请从首个单元开始顺序执行。


#### 6.1.1 定义核函数并绑定 GM

每个 tile 处理 256 个 float。入口调用 `InitSocState()`，随后按核数均分数据，通过核索引得到各核的起始偏移，并绑定三个 GlobalTensor。本例每核处理 2048 个元素。


In [ ]:
%%writefile Sources/02.05/add_static.asc
#include "kernel_operator.h"
#include <cstdint>

constexpr uint32_t TILE_LENGTH = 256;

// 固定 tile 的入门实现：Host 保证每核长度是 TILE_LENGTH 的整数倍。
__global__ __vector__ void vector_static(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength)
{
    AscendC::InitSocState();
    const uint32_t blockLength = totalLength / AscendC::GetBlockNum();
    const uint32_t blockOffset = blockLength * AscendC::GetBlockIdx();
    AscendC::GlobalTensor<float> xGm, yGm, zGm;
    xGm.SetGlobalBuffer((__gm__ float *)x + blockOffset, blockLength);
    yGm.SetGlobalBuffer((__gm__ float *)y + blockOffset, blockLength);
    zGm.SetGlobalBuffer((__gm__ float *)z + blockOffset, blockLength);



#### 6.1.2 线性分配 UB 空间

在循环外创建唯一的 UB 分配器，依次分配两个输入 Tensor 和一个输出 Tensor，每个容纳 256 个 float，合计 3072 字节。后续 8 轮循环复用这三个 Tensor。


In [ ]:
%%writefile -a Sources/02.05/add_static.asc
    // 同一 kernel 中，UB 只使用一个线性分配器；在循环外一次性规划空间。
    AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    auto xLocal = ubAllocator.Alloc<float, TILE_LENGTH>();
    auto yLocal = ubAllocator.Alloc<float, TILE_LENGTH>();
    auto zLocal = ubAllocator.Alloc<float, TILE_LENGTH>();



#### 6.1.3 循环搬入数据

每核循环 8 次，每次将 GM 偏移增加 256 个元素。首轮可直接搬入；后续轮次先等待 `V_MTE2`，确保上轮计算已经读完输入。搬入后通过 `MTE2_V` 保证 Vector 读取到完整数据。


In [ ]:
%%writefile -a Sources/02.05/add_static.asc
    const uint32_t loopCount = blockLength / TILE_LENGTH;
    for (uint32_t i = 0; i < loopCount; ++i) {
        const uint32_t offset = i * TILE_LENGTH;
        if (i != 0) {
            // 上轮 Vector 读完输入，才允许本轮搬入覆盖输入。
            AscendC::WaitFlag<AscendC::HardEvent::V_MTE2>(EVENT_ID0);
        }
        AscendC::DataCopy(xLocal, xGm[offset], TILE_LENGTH);
        AscendC::DataCopy(yLocal, yGm[offset], TILE_LENGTH);
        AscendC::SetFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);



#### 6.1.4 执行加法计算

从第二轮开始，先等待 `MTE3_V`，确保上轮输出已被搬出，再通过 `Add` 写入当前结果。若还有下一轮，则设置 `V_MTE2`，允许下一轮覆盖输入 buffer。


In [ ]:
%%writefile -a Sources/02.05/add_static.asc
        if (i != 0) {
            // 上轮 MTE3 读完输出，才允许本轮 Vector 覆盖输出。
            AscendC::WaitFlag<AscendC::HardEvent::MTE3_V>(EVENT_ID0);
        }
        AscendC::Add(zLocal, xLocal, yLocal, TILE_LENGTH);
        if (i + 1 < loopCount) {
            AscendC::SetFlag<AscendC::HardEvent::V_MTE2>(EVENT_ID0);
        }


#### 6.1.5 搬出结果并准备复用

通过 `V_MTE3` 等待计算完成，再将当前 tile 写回 GM。若还有下一轮，则设置 `MTE3_V`，使下一轮计算能够在本轮搬出完成后复用输出 buffer。


In [ ]:
%%writefile -a Sources/02.05/add_static.asc
        AscendC::SetFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);
        AscendC::DataCopy(zGm[offset], zLocal, TILE_LENGTH);
        if (i + 1 < loopCount) {
            AscendC::SetFlag<AscendC::HardEvent::MTE3_V>(EVENT_ID0);
        }
    }


#### 6.1.6 结束核函数

循环结束后，调用 `PipeBarrier<PIPE_ALL>()` 等待流水完成。接下来将 Host 调用代码追加到同一文件。


In [ ]:
%%writefile -a Sources/02.05/add_static.asc
    AscendC::PipeBarrier<PIPE_ALL>();
}



### 6.2 Host 调用与精度验证

Host 侧通过 AscendCL 申请 GM、复制输入、启动 kernel、等待完成、回拷结果和释放资源。将以下代码追加到 `add_static.asc`，形成完整的可执行程序。

用例固定为 8 核、每核 8 个 tile，每个 tile 包含 256 个 float，共 16384 个元素。


In [ ]:
%%writefile -a Sources/02.05/add_static.asc
#include "acl/acl.h"
#include <algorithm>
#include <cmath>
#include <cstdlib>
#include <iostream>
#include <vector>

inline void CheckAcl(aclError code, const char *call)
{
    if (code != ACL_SUCCESS) {
        std::cerr << "[ACL ERROR] " << call << ": " << code << std::endl;
        std::exit(EXIT_FAILURE);
    }
}
#define CHECK_ACL(call) CheckAcl((call), #call)

// deviceId 可通过唯一的位置参数指定，默认使用设备 0。
int main(int argc, char **argv)
{
    int deviceId = 0;
    if (argc > 2) {
        std::cerr << "Usage: " << argv[0] << " [device_id]" << std::endl;
        return 2;
    }
    if (argc == 2) {
        char *end = nullptr;
        long parsed = std::strtol(argv[1], &end, 10);
        if (argv[1][0] == '\0' || *end != '\0' || parsed < 0 || parsed > 255) {
            std::cerr << "Invalid device_id" << std::endl;
            return 2;
        }
        deviceId = static_cast<int>(parsed);
    }
    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    aclrtStream stream = nullptr;
    CHECK_ACL(aclrtCreateStream(&stream));

    constexpr uint32_t numBlocks = 8;
    constexpr uint32_t tilesPerCore = 8;
    const uint32_t totalLength = numBlocks * tilesPerCore * TILE_LENGTH;
    const size_t bytes = totalLength * sizeof(float);
    std::vector<float> x(totalLength), y(totalLength), output(totalLength);
    // 随索引、tile 和核变化；二进制可精确表示的值便于定位搬运/同步错误。
    for (uint32_t i = 0; i < totalLength; ++i) {
        x[i] = static_cast<float>(static_cast<int>(i % 1021) - 510) * 0.125f;
        y[i] = static_cast<float>(static_cast<int>((i * 17) % 509) - 254) * 0.0625f;
    }
    void *xDevice = nullptr, *yDevice = nullptr, *zDevice = nullptr;
    CHECK_ACL(aclrtMalloc(&xDevice, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(&yDevice, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(&zDevice, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMemcpy(xDevice, bytes, x.data(), bytes, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(yDevice, bytes, y.data(), bytes, ACL_MEMCPY_HOST_TO_DEVICE));
    // 未写出的输出保留非有限值，避免误将未计算的数据判为正确。
    CHECK_ACL(aclrtMemset(zDevice, bytes, 0xff, bytes));
    vector_static<<<numBlocks, nullptr, stream>>>(
        static_cast<uint8_t *>(xDevice), static_cast<uint8_t *>(yDevice),
        static_cast<uint8_t *>(zDevice), totalLength);
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(output.data(), bytes, zDevice, bytes, ACL_MEMCPY_DEVICE_TO_HOST));
    bool passed = true;
    float maxError = 0.0f;
    for (uint32_t i = 0; i < totalLength; ++i) {
        const float expected = x[i] + y[i];
        const float error = std::fabs(output[i] - expected);
        maxError = std::max(maxError, error);
        if (!std::isfinite(output[i]) || error > 1e-6f) {
            if (passed) {
                std::cerr << "Mismatch at " << i << ": got " << output[i]
                          << ", expected " << expected << std::endl;
            }
            passed = false;
        }
    }
    std::cout << (passed ? "[PASS] " : "[FAIL] ")
              << "Add"
              << " blocks=" << numBlocks << " tiles/core=" << tilesPerCore
              << " elements=" << totalLength << " max_error=" << maxError << std::endl;
    CHECK_ACL(aclrtFree(xDevice));
    CHECK_ACL(aclrtFree(yDevice));
    CHECK_ACL(aclrtFree(zDevice));
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return passed ? 0 : 1;
}


### 6.3 CMake 配置

构建目标 `demo` 使用 A2/A3 对应的 `--npu-arch=dav-2201`。

`KERNEL_SOURCE` 默认为 Add，也可指定习题或参考答案文件。

In [ ]:
%%writefile Sources/02.05/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
find_package(ASC REQUIRED)
project(static_tensor_vector LANGUAGES ASC CXX)

# 默认编译 Add；习题和答案通过 -DKERNEL_SOURCE=/absolute/path/sub_static.asc 指定。
set(KERNEL_SOURCE "${CMAKE_CURRENT_SOURCE_DIR}/add_static.asc" CACHE FILEPATH "Kernel source")
add_executable(demo "${KERNEL_SOURCE}")
target_compile_options(demo PRIVATE $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>)


### 6.4 编译和运行

In [ ]:
!cmake -S Sources/02.05 -B Sources/02.05/build_add && \
    cmake --build Sources/02.05/build_add -j2 && \
    ./Sources/02.05/build_add/demo


程序预期输出：

```text
[PASS] Add blocks=8 tiles/core=8 elements=16384 max_error=0
```

## 7. 本节小结

本节以 Add 算子为例，完成了静态 Tensor 方式的 Vector 算子开发：

- **线性分配本地内存**：使用一个 `LocalMemAllocator<Hardware::UB>` 在循环外分配输入、输出 LocalTensor，并在循环中复用。
- **组织数据流**：通过 `GlobalTensor` 绑定各核负责的 GM 区域，按 `DataCopy → Add → DataCopy` 完成搬入、计算和搬出。
- **管理流水同步**：根据数据依赖，使用 `SetFlag` 设置同步事件、`WaitFlag` 等待同步事件，协调不同流水的执行顺序，保证数据访问和 buffer 复用安全。
- **处理入口与退出**：kernel 入口调用 `InitSocState()`，退出前通过 `PipeBarrier<PIPE_ALL>()` 等待流水完成。


## 8. 课后实践：静态 Tensor 减法

实现 `z = x - y`，输入输出均为 float、ND 布局，shape 为 `(8, 2048)`。使用 8 核、每核 8 个 tile，每 tile 256 个元素。

要求：

1. 在 kernel 入口初始化 SoC 状态，计算本核 GM 偏移并绑定输入输出。
2. **必须使用静态 Tensor 方式分配 LocalTensor**：使用一个 `LocalMemAllocator<Hardware::UB>` 线性分配器；禁止改用 TPipe/TQue/TBuf。
3. 在循环外规划三个 UB buffer，循环内完成搬入、`Sub`、搬出。
4. 补齐四个方向的事件同步，正确处理首轮、末轮。
5. 退出前执行全流水屏障，使用提供的 Host 驱动逐元素验证。

请完成线性分配与同步逻辑，并验证计算结果。

下面写入待完成模板，**重新运行该单元会覆盖之前生成的习题文件**；请在该代码单元中完成实现并重新执行，或直接编辑文件后跳过该单元。

In [ ]:
%%writefile Sources/02.05/sub_static.asc
#include "kernel_operator.h"
#include <cstdint>

constexpr uint32_t TILE_LENGTH = 256;

__global__ __vector__ void vector_static(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength)
{
    // TODO 1：入口调用 InitSocState，绑定当前核的三个 GlobalTensor。
    // TODO 2：在循环外通过一个 UB LocalMemAllocator 分配 xLocal/yLocal/zLocal；
    // TODO 3：循环内完成 DataCopy -> Sub -> DataCopy。
    // TODO 4：补齐 MTE2_V、V_MTE3 以及跨轮 V_MTE2、MTE3_V 同步。
    // TODO 5：退出前等待 PIPE_ALL。
    // 完成实现后删除下一行；禁止使用 TPipe、TQue 或 TBuf 替代静态分配。
#error "Complete the static Tensor Sub exercise before building"
}

#include "acl/acl.h"
#include <algorithm>
#include <cmath>
#include <cstdlib>
#include <iostream>
#include <vector>

inline void CheckAcl(aclError code, const char *call)
{
    if (code != ACL_SUCCESS) {
        std::cerr << "[ACL ERROR] " << call << ": " << code << std::endl;
        std::exit(EXIT_FAILURE);
    }
}
#define CHECK_ACL(call) CheckAcl((call), #call)

// deviceId 可通过唯一的位置参数指定，默认使用设备 0。
int main(int argc, char **argv)
{
    int deviceId = 0;
    if (argc > 2) {
        std::cerr << "Usage: " << argv[0] << " [device_id]" << std::endl;
        return 2;
    }
    if (argc == 2) {
        char *end = nullptr;
        long parsed = std::strtol(argv[1], &end, 10);
        if (argv[1][0] == '\0' || *end != '\0' || parsed < 0 || parsed > 255) {
            std::cerr << "Invalid device_id" << std::endl;
            return 2;
        }
        deviceId = static_cast<int>(parsed);
    }
    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    aclrtStream stream = nullptr;
    CHECK_ACL(aclrtCreateStream(&stream));

    constexpr uint32_t numBlocks = 8;
    constexpr uint32_t tilesPerCore = 8;
    const uint32_t totalLength = numBlocks * tilesPerCore * TILE_LENGTH;
    const size_t bytes = totalLength * sizeof(float);
    std::vector<float> x(totalLength), y(totalLength), output(totalLength);
    // 随索引、tile 和核变化；二进制可精确表示的值便于定位搬运/同步错误。
    for (uint32_t i = 0; i < totalLength; ++i) {
        x[i] = static_cast<float>(static_cast<int>(i % 1021) - 510) * 0.125f;
        y[i] = static_cast<float>(static_cast<int>((i * 17) % 509) - 254) * 0.0625f;
    }
    void *xDevice = nullptr, *yDevice = nullptr, *zDevice = nullptr;
    CHECK_ACL(aclrtMalloc(&xDevice, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(&yDevice, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(&zDevice, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMemcpy(xDevice, bytes, x.data(), bytes, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(yDevice, bytes, y.data(), bytes, ACL_MEMCPY_HOST_TO_DEVICE));
    // 未写出的输出保留非有限值，避免误将未计算的数据判为正确。
    CHECK_ACL(aclrtMemset(zDevice, bytes, 0xff, bytes));
    vector_static<<<numBlocks, nullptr, stream>>>(
        static_cast<uint8_t *>(xDevice), static_cast<uint8_t *>(yDevice),
        static_cast<uint8_t *>(zDevice), totalLength);
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(output.data(), bytes, zDevice, bytes, ACL_MEMCPY_DEVICE_TO_HOST));
    bool passed = true;
    float maxError = 0.0f;
    for (uint32_t i = 0; i < totalLength; ++i) {
        const float expected = x[i] - y[i];
        const float error = std::fabs(output[i] - expected);
        maxError = std::max(maxError, error);
        if (!std::isfinite(output[i]) || error > 1e-6f) {
            if (passed) {
                std::cerr << "Mismatch at " << i << ": got " << output[i]
                          << ", expected " << expected << std::endl;
            }
            passed = false;
        }
    }
    std::cout << (passed ? "[PASS] " : "[FAIL] ")
              << "Sub"
              << " blocks=" << numBlocks << " tiles/core=" << tilesPerCore
              << " elements=" << totalLength << " max_error=" << maxError << std::endl;
    CHECK_ACL(aclrtFree(xDevice));
    CHECK_ACL(aclrtFree(yDevice));
    CHECK_ACL(aclrtFree(zDevice));
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return passed ? 0 : 1;
}


完成模板并删除其中的 `#error` 后，执行下面的命令编译运行。


In [ ]:
!cmake -S Sources/02.05 -B Sources/02.05/build_practice -DKERNEL_SOURCE="$PWD/Sources/02.05/sub_static.asc" && \
    cmake --build Sources/02.05/build_practice -j2 && \
    ./Sources/02.05/build_practice/demo


参考答案及完整源码位于 [answer/02.05](./answer/02.05/answers.md)。执行下面的单元查看并编译运行参考答案。


In [ ]:
!cat answer/02.05/sub_static.asc
!cmake -S Sources/02.05 -B Sources/02.05/build_answer -DKERNEL_SOURCE="$PWD/answer/02.05/sub_static.asc" && \
    cmake --build Sources/02.05/build_answer -j2 && \
    ./Sources/02.05/build_answer/demo
